# Solve the volume relationships for the Cao et al. (2019) rootstocks

Using your clarification:

- `RWCtlp (%)` and `ROWCtlp (%)` are percentages and should be converted to decimals.
- `Va/Vp` is already a ratio and should **not** be divided by 100.

The equations are:

$$
\mathrm{RWC}_{tlp} = \frac{V_t - V_e}{V_t}, \quad
\mathrm{ROWC}_{tlp} = \frac{V_o - V_e}{V_o}, \quad
\frac{V_a}{V_p} = \frac{V_t - V_o}{V_t}
$$

From these,

$$
V_e = V_t(1 - \mathrm{RWC}_{tlp}), \quad
V_o = V_t\left(1 - \frac{V_a}{V_p}\right)
$$

and consistency requires:

$$
1 - \mathrm{RWC}_{tlp} = \left(1 - \frac{V_a}{V_p}\right)(1 - \mathrm{ROWC}_{tlp})
$$

So the data can determine **relative** values of $V_o$ and $V_e$ with respect to $V_t$, but not a unique absolute triplet $(V_t, V_o, V_e)$ unless one extra scale constraint is supplied. In the code below, I report the exact normalized solution with $V_t = 1$ and also check whether each row is consistent with all three equations.

From USFS Specific Gravity and Other Properties of Wood and Bark for 156 Tree Species Found in North America (2009) S.G. of Pecan - green volume basis dry weight - is 0.60. Thus wood density (WD) ~ 0.600 g/cm3. From Christofferson (2016) and Siau (1984): Vt = 1 - WD/1.54.   

Estimate for Vt: 0.61 cm3/cm3.   
   
Effective volume of tree stem: ~140 cm3 (based on DBH and canopy height of lab seedlings 6 and 9. **Right now, this assumes the total tree area is sapwood**)   
   
Effective water storage volume of tree stem at saturation: 85.4 cm3
   
~~Taking the average leaf area from the seedlings in Cao et al (2019): 25.4 cm2~~
~~Total storage water content on a leaf area basis ~ 3.36 cm = 0.036 m~~ 

Taking the average leaf area from the seedlings in the lab:
80.0 cm2 

Total storage water content on a leaf area basis ~ 1.065 cm = 0.011 m
  


In [3]:
from pathlib import Path
import shutil
import tempfile

import numpy as np
import pandas as pd

sheet_name = "Table1_PV_data"


def resolve_workbook_path() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd / "literature_data" / "Cao et al 2019 Data.xlsx",
        cwd / "Cao et al 2019 Data.xlsx",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    attempted = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        "Could not find 'Cao et al 2019 Data.xlsx'. Tried:\n" + attempted
    )


def read_excel_with_fallback(path: Path, sheet_name: str) -> pd.DataFrame:
    try:
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    except PermissionError:
        temp_path = Path(tempfile.gettempdir()) / path.name
        shutil.copy2(path, temp_path)
        return pd.read_excel(temp_path, sheet_name=sheet_name, engine="openpyxl")


def percent_to_decimal(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values / 100.0


workbook = resolve_workbook_path()
df = read_excel_with_fallback(workbook, sheet_name)

required_columns = ["Rootstock", "RWCtlp (%)", "ROWCtlp (%)", "Va/Vp"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

pv = df[required_columns].copy()
pv["RWCtlp_decimal"] = percent_to_decimal(pv["RWCtlp (%)"])
pv["ROWCtlp_decimal"] = percent_to_decimal(pv["ROWCtlp (%)"])
pv["Va_Vp_ratio"] = pd.to_numeric(pv["Va/Vp"], errors="coerce")

# Exact normalized solution for the family of solutions parameterized by Vt.
# Setting Vt = 1 gives the relative volumes directly.
pv["Vt_normalized"] = 1.0
pv["Vo_normalized"] = 1.0 - 1 / pv["Va_Vp_ratio"]
pv["Ve_normalized_from_RWC"] = 1.0 - pv["RWCtlp_decimal"]
pv["Ve_normalized_from_ROWC_and_VaVp"] = pv["Vo_normalized"] * (1.0 - pv["ROWCtlp_decimal"])

pv["consistency_error"] = (
    pv["Ve_normalized_from_RWC"] - pv["Ve_normalized_from_ROWC_and_VaVp"]
)
pv["all_equations_consistent"] = np.isclose(pv["consistency_error"], 0.0, atol=1e-9)
pv["Vo_is_positive"] = pv["Vo_normalized"] > 0.0

result = pv[
    [
        "Rootstock",
        "RWCtlp_decimal",
        "ROWCtlp_decimal",
        "Va_Vp_ratio",
        "Vt_normalized",
        "Vo_normalized",
        "Ve_normalized_from_RWC",
        "Ve_normalized_from_ROWC_and_VaVp",
        "consistency_error",
        "all_equations_consistent",
        "Vo_is_positive",
    ]
]

print(f"Using workbook: {workbook}")
print("Exact normalized solution assuming Vt = 1.0")
print("Absolute values remain undetermined without one extra scale constraint.")
print()
print(result.to_string(index=False))

Using workbook: c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\literature_data\Cao et al 2019 Data.xlsx
Exact normalized solution assuming Vt = 1.0
Absolute values remain undetermined without one extra scale constraint.

 Rootstock  RWCtlp_decimal  ROWCtlp_decimal  Va_Vp_ratio  Vt_normalized  Vo_normalized  Ve_normalized_from_RWC  Ve_normalized_from_ROWC_and_VaVp  consistency_error  all_equations_consistent  Vo_is_positive
 87MX1-2.2          0.7731           0.5319         1.23            1.0       0.186992                  0.2269                          0.087531           0.139369                     False            True
 87MX5-1.7          0.8058           0.4802         2.44            1.0       0.590164                  0.1942                          0.306767          -0.112567                     False            True
   Elliott          0.8587           0.5764         2.31            1.0       0.567100                  0.1413    